# CRISPR Screen: Waterfall and Pie Charts

Referencing the `CRISPR_Volcano_Analysis.ipynb` in the same directory, this notebook only generates two figures:
- Waterfall Plot (all sgRNA log2FC values sorted)
- Gene Effects distribution pie chart (binned by log2FC)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style and fonts
sns.set_palette('Greens')
plt.style.use('seaborn-v0_8-whitegrid')
print('✓ Imports ready')

In [ ]:
# Use the same data source as the reference notebook
DATA_FILE = '/path/to/CRISPR/Data/TGEV-筛选结果分析-2023-12-5.xls'
SHEET_NAME = 'zlu_lib'

# Load data
xl = pd.ExcelFile(DATA_FILE)
print('Available sheets:', xl.sheet_names)

df = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)
print(f'Loaded sheet: {SHEET_NAME}, shape={df.shape}')
print('Columns:', list(df.columns))

# Keep only the columns needed for plotting
assert 'log2(Fold_change)' in df.columns, 'Required column log2(Fold_change) does not exist'
log2fc = df['log2(Fold_change)'].astype(float).copy()

In [ ]:
# Classification thresholds and colors (consistent with Quadrant_Classification_R.ipynb)
# bins: Depleted (<0), Slightly Enriched (0<=x<2), Moderately Enriched (2<=x<5), Highly Enriched (>=5)
BINS = [-np.inf, 0, 2, 5, np.inf]
LABELS = ['Depleted', 'Slightly Enriched', 'Moderately Enriched', 'Highly Enriched']
COLORS = {
    'Depleted': '#4472C4',             # Blue (Depleted)
    'Slightly Enriched': '#FFD700',    # Yellow (Slightly Enriched)
    'Moderately Enriched': '#FF8C00',  # Orange (Moderately Enriched)
    'Highly Enriched': '#DC143C',      # Red (Highly Enriched)
}

cats = pd.cut(log2fc, bins=BINS, labels=LABELS, right=False)
df_plot = pd.DataFrame({'log2FC': log2fc, 'category': cats})
print(df_plot['category'].value_counts().sort_index())

In [ ]:
# sgRNA basic info + category labels + log2(Fold_change)
from pathlib import Path

# Select available columns
cols = [c for c in ['sgRNA','seq','GeneID_x','GeneID_y','Fold_change','log2(Fold_change)'] if c in df.columns]
sgRNA_table = df[cols].copy()

# Append classification results (same index as df)
sgRNA_table['category'] = df_plot['category'].astype(str).values

# Sort (by log2FC descending)
if 'log2(Fold_change)' in sgRNA_table.columns:
    sgRNA_table = sgRNA_table.sort_values('log2(Fold_change)', ascending=False)

# Display top 20 rows
print('sgRNA info with category (top 20):')
from IPython.display import display
display(sgRNA_table.head(20))

# Optional: save CSV
SAVE_TABLE = True
results_dir = '/path/to/CRISPR/results'
if SAVE_TABLE:
    Path(results_dir).mkdir(parents=True, exist_ok=True)
    out_csv = f"{results_dir}/sgRNA_info_with_category_{SHEET_NAME}.csv"
    sgRNA_table.to_csv(out_csv, index=False)
    print('✓ Saved:', out_csv)

In [ ]:
# Waterfall Plot
# Sort and color by category
sorted_df = df_plot.sort_values('log2FC', ascending=False).reset_index(drop=True)
colors = sorted_df['category'].map(COLORS).values

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(range(len(sorted_df)), sorted_df['log2FC'].values, color=colors, edgecolor='none')

# Reference lines: 0, 2, 5
for y, ls in [(0, '--'), (2, ':'), (5, ':')]:
    ax.axhline(y=y, color='gray', linestyle=ls, linewidth=1)

ax.set_title('Waterfall Plot: All sgRNA Effects', fontsize=12)
ax.set_xlabel('Ranked sgRNAs')
ax.set_ylabel('log2(Fold Change)')

# Build legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=COLORS[k], label=k) for k in LABELS]
ax.legend(handles=legend_patches, loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Pie Chart: Distribution of Gene Effects
counts = df_plot['category'].value_counts().reindex(LABELS).fillna(0)
labels = [
    'Depleted\n(log2FC<0)',
    'Slightly Enriched\n(0\u2264log2FC<2)',
    'Moderately Enriched\n(2\u2264log2FC<5)',
    'Highly Enriched\n(log2FC\u22655)'
]

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=[COLORS[k] for k in LABELS],
    textprops={'color': 'black'}
)
ax.set_title('Distribution of Gene Effects', fontsize=12)
ax.axis('equal')  # Keep as circle
plt.tight_layout()
plt.show()

In [ ]:
# Optional: save figures to results directory
SAVE = True
RESULTS_DIR = '/path/to/CRISPR/results'
PREFIX = 'CRISPR_waterfall_pie'

if SAVE:
    # Re-draw and save to avoid capturing from previous cell
    # Waterfall
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(range(len(sorted_df)), sorted_df['log2FC'].values, color=colors, edgecolor='none')
    for y, ls in [(0, '--'), (2, ':'), (5, ':')]:
        ax.axhline(y=y, color='gray', linestyle=ls, linewidth=1)
    ax.set_title('Waterfall Plot: All sgRNA Effects', fontsize=12)
    ax.set_xlabel('Ranked sgRNAs')
    ax.set_ylabel('log2(Fold Change)')
    from matplotlib.patches import Patch
    legend_patches = [Patch(facecolor=COLORS[k], label=k) for k in LABELS]
    ax.legend(handles=legend_patches, loc='upper right', frameon=True)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/{PREFIX}_waterfall_{SHEET_NAME}.pdf", dpi=300)
    plt.close(fig)

    # Pie
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(counts.values, labels=labels, autopct='%1.1f%%', startangle=90,
           colors=[COLORS[k] for k in LABELS], textprops={'color': 'black'})
    ax.set_title('Distribution of Gene Effects', fontsize=12)
    ax.axis('equal')
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/{PREFIX}_pie_{SHEET_NAME}.pdf", dpi=300)
    plt.close(fig)

print('✓ Done. Figures displayed and saved.' if SAVE else '✓ Done. Figures displayed.')

In [ ]:
# Check if there are missing sgRNAs in the original Excel file
print('=== Checking Raw Data ===')
print(f'Rows read from original Excel file: {df.shape[0]}')
print(f'Number of unique sgRNAs: {df["sgRNA"].nunique()}')
print(f'Any duplicate sgRNAs: {df["sgRNA"].duplicated().any()}')

# Check for null values
print(f'\nNumber of null values in sgRNA column: {df["sgRNA"].isna().sum()}')

# Check data volume for all sheets
xl = pd.ExcelFile(DATA_FILE)
print('\n=== Row Count per Sheet ===')
for sheet in xl.sheet_names:
    temp_df = pd.read_excel(DATA_FILE, sheet_name=sheet)
    print(f'{sheet}: {len(temp_df)} rows')
    
print('\nIf the zlu_lib sheet in the original Excel file indeed has only 120 rows, then the output CSV file is correct.')
print('If you believe there should be 121 sgRNAs, please check:')
print('1. Is it in another sheet?')
print('2. Was it lost before data processing?')
print('3. Was your library design originally 120 sgRNAs?')